In [0]:
from pyspark.sql.types import StringType , StructField, StructType

VOL_PATH = "/Volumes/ecommerce/raw/raw_aws_data/order_items/landing"
# Read all csv's and create Bronze table of it . 

schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",         StringType(), True)
])
df_orders = spark.read.csv(VOL_PATH, schema = schema , header=True, inferSchema=True)

df_orders.printSchema

In [0]:
# write it as Bronze dataset 
df_orders.write.format("delta").mode("overwrite").saveAsTable('ecommerce.bronze.brz_orders')

#### We have to add few columns to it before we put it in SIlver 

In [0]:
%sql
select * from ecommerce.bronze.brz_orders limit 10 

In [0]:
# we have to replace Two - 2 , remove $ from unit_price , remove % from discount_pct 

from pyspark.sql.functions import col, when, regexp_replace, lower, to_date
df_orders = spark.read.table('ecommerce.bronze.brz_orders')
df_orders = df_orders.withColumns({
    'quantity' : when(col('quantity') == 'Two', 2).otherwise(col('quantity')).cast("int"),
    'unit_price': regexp_replace(col("unit_price"), "[$]", ""),
    'discount_pct' : regexp_replace(col("discount_pct"), "[%]", "").cast("double"),
    'coupon_code' : when (col("coupon_code").isNotNull(), lower(col('coupon_code'))).otherwise(None),
    'channel' : when(col('channel') == 'app', 'mobile_app').when(col('channel') == 'web', 'website').otherwise('na'),
    'dt'      : to_date(col('dt'),'yyyy-MM-dd')
                                 })
                                 
display(df_orders.limit(10))

In [0]:
df_orders.write \
    .format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .save("s3://sj-dbr-demo-proj/silver_data/slv_orders")

spark.sql("""
          create table if not exists ecommerce.silver.slv_orders
          using delta
          location "s3://sj-dbr-demo-proj/silver_data/slv_orders"
          """)

In [0]:
dbutils.notebook.exit("SUCCESS")